# Qwen3 0.6B — fixed four-pair Dual-KL merger (Kaggle T4 ×2)

**User-specified experiment:** merge original zero-based layers **(2,3), (4,5), (6,7), (8,9)** into four replacement layers. The other 20 layers, embeddings, norms and LM head remain frozen. Train only the shared merger network with **Hidden MSE + Local Hidden Feature KL + Final Vocabulary KL + Correction**, for **10 complete epochs**, **10,000 merger-training token occurrences per language** across seven languages.

No CE acceptance cutoff, no automatic merge-pair selection, no storage gate; CE is reported only as a diagnostic. Exactly the existing fast paired SFT setup remains: **200 examples/language**, 1 epoch, original FP16 LoRA vs merged NF4 QLoRA, followed by the same 30-question/language MMMLU/INCLUDE/MGSM pilot. Use **T4 ×2** and Internet. Training duration is not guaranteed.

**Fresh output directory:** this notebook does not modify the prior 10- or 20-epoch experiments. Existing saved models from those runs should not be reused for the changed objective.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json,importlib.metadata
ROOT=Path('/kaggle/working');REPO=ROOT/'Layer-Merging-FixedDualKL';OUTPUT=ROOT/'outputs/qwen3_8block_fixed_2_4_6_8_dualKL_10epochs'
if not REPO.exists():subprocess.run(['git','clone','https://github.com/amritosanto/Layer-Merging.git',str(REPO)],check=True)
else:subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
assert 'benchmark-worker' in (REPO/'run.py').read_text(), 'Repo does not contain the GPU isolation fix'
print('Checked out:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
packages={'transformers':'4.51.3','peft':'0.15.2','datasets':'4.4.1','accelerate':'1.6.0','bitsandbytes':'0.48.2','huggingface-hub':'0.36.0','tokenizers':'0.21.4'}
missing=[]
for name,version in packages.items():
    try:current=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:current=None
    if current!=version:missing.append(f'{name}=={version}')
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Retained existing CUDA PyTorch; compatible dependencies installed')

In [ ]:
import torch
assert torch.cuda.device_count()>=2,'Enable Kaggle T4 x2'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
config=json.loads((REPO/'config.smoke_fixed4_dualkl.json').read_text());config['output_dir']=str(OUTPUT)
assert config['merger_blocks']==8 and config['sft_epochs']==1
assert config['epochs']==10 and config['fixed_merge_pairs']==[2,4,6,8]
assert config['min_merged_pairs']==config['max_merged_pairs']==4
assert config['local_kl_weight']==1.0 and config['local_kl_temperature']==2.0
assert config['max_optimizer_steps']==0 and config['training_hours']==0
assert config['sft_examples_per_language']==200
assert config['pilot_questions']==30 and config['max_relative_ce_increase'] is None
assert config['train_tokens_per_language']==10000
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG=ROOT/'qwen-fixed-four-pair-dualkl-config.json'
CONFIG.write_text(json.dumps(config,indent=2))
ENV=os.environ.copy();ENV.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false')
def run_stage(stage):
    subprocess.run([sys.executable,'run.py',stage,'--config',str(CONFIG)],cwd=REPO,env=ENV,check=True)

## Prepare data and verify the four-term loss
The quick CPU check confirms that local KL is zero for identical teacher/student hidden states, gives finite nonzero gradients for different states, and that the fixed-plan settings validate.

In [ ]:
import torch, sys
sys.path.insert(0,str(REPO))
from config import Config
from src.objectives import local_hidden_distribution_kl
runtime_cfg=Config.load(CONFIG)
assert tuple(runtime_cfg.fixed_merge_pairs)==(2,4,6,8)
x=torch.randn(2,3,128)
assert abs(float(local_hidden_distribution_kl(x,x,2.0)))<1e-5
y=(x.detach()+torch.randn_like(x)*0.5).requires_grad_()
kl=local_hidden_distribution_kl(y,x,2.0)
kl.backward()
assert float(kl)>0 and y.grad is not None and torch.isfinite(y.grad).all() and y.grad.abs().sum()>0
print('CPU loss tests passed; fixed pairs:',tuple(runtime_cfg.fixed_merge_pairs))
run_stage('prepare')
run_stage('preflight')

## Train only the merger network for ten full epochs
All four selected pairs are optimized *together* on every batch. Teacher, original student layers, unmerged layers, embeddings and LM head are frozen. The four loss components appear separately in training logs. Checkpoints resume only this new configuration.

In [ ]:
run_stage('train')

In [ ]:
run_stage('export')
import json
export=json.loads((OUTPUT/'manifests/final_export.json').read_text())
assert export['pairs']==[2,4,6,8] and export['remaining_layers']==24
assert export['max_relative_ce_increase'] is None and export['ce_target_met'] is None
print('Fixed merged layer pairs:',export['original_layer_pairs'])
print('Diagnostic ranking CE:',export['ranking_ce'])
print('No CE threshold or automatic layer selection used')

## Paired SFT — unchanged
Original FP16 LoRA vs fixed-merged NF4 QLoRA, each receiving the same audited training records and response-only labels. One epoch, 200 examples per language. No additional SFT changes.

In [ ]:
run_stage('sft-prepare')

In [ ]:
run_stage('sft-both')

In [ ]:
run_stage('sft-verify')

## Post-SFT benchmarks using separate T4 worker processes
Run the existing MMMLU, INCLUDE and MGSM evaluation protocols, 30 questions/language. The two GPU workers are isolated to avoid the previous PEFT meta-tensor race.

In [ ]:
run_stage('benchmark')

In [ ]:
run_stage('report')
from IPython.display import display,FileLink
display(FileLink(str(OUTPUT/'REPORT.md')))

## Download a compact diagnostic archive
This ZIP includes reports, manifests, merger-training logs, benchmark predictions, charts, and SFT metrics. It intentionally excludes model/checkpoint weights to avoid the previous Kaggle disk-space ZIP failure. Save the notebook as a Kaggle version with outputs to retain model weights.

In [ ]:
from pathlib import Path
import zipfile
from IPython.display import FileLink,display
archive=ROOT/'Qwen3_FixedPairs_DualKL_10Epoch_DIAGNOSTICS.zip'
assert (OUTPUT/'REPORT.md').exists(),'Finish the report stage before archiving'
wanted=['REPORT*.md','manifests/*.json','rankings/*.json','rankings/*.csv',
        'logs/**/*.json','logs/**/*.csv','benchmark_results/**/*.jsonl',
        'benchmark_results/**/*.csv','benchmark_results/**/*.json','charts/*.png',
        'sft/*/training_metrics.json','sft/*/preflight*.json','sft/dataset_manifest.json']
files=sorted({p for pat in wanted for p in OUTPUT.glob(pat) if p.is_file()})
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in files:z.write(p,p.relative_to(OUTPUT))
print('Diagnostic files:',len(files),'ZIP MB:',round(archive.stat().st_size/(1024**2),2))
print('Weights remain in:',OUTPUT)
display(FileLink(str(archive)))